<a href="https://colab.research.google.com/github/Douglasmach/modelagemComputacional/blob/main/modelagemComp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install cdsapi

import cdsapi

dataset = "reanalysis-era5-single-levels"
request = {
    "product_type": ["reanalysis"],
    "variable": [
        "10m_u_component_of_wind",
        "10m_v_component_of_wind"
    ],
    "year": ["2015"],
    "month": ["04"],
    "day": [
        "02", "03", "04",
        "05"
    ],
    "time": [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": [-23.5, -46.5, -24.25, -46.25]
}

client = cdsapi.Client(
    url = 'https://cds.climate.copernicus.eu/api',
    key = '9394dee4-aa76-4795-942b-2f6f6e677215')
client.retrieve(dataset, request).download()


2026-10-05 19:53:57,672 INFO Request ID is 601260a5-937d-4a9f-af46-c01bc9832350
INFO:ecmwf.datastores.legacy_client:Request ID is 601260a5-937d-4a9f-af46-c01bc9832350
2026-10-05 19:53:59,556 INFO status has been updated to successful
INFO:ecmwf.datastores.legacy_client:status has been updated to successful


6995fb773ec93a93b2827cad907199e1.nc:   0%|          | 0.00/39.6k [00:00<?, ?B/s]

'6995fb773ec93a93b2827cad907199e1.nc'

In [5]:
# ==============================================================================
# ETAPA 1: Instalação de Dependências e Download dos Pesos do SAM
# ==============================================================================
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!pip install -q opencv-python matplotlib wget

import os
import glob
import wget
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
from segment_anything import sam_model_registry, SamPredictor

# 1. Download dos pesos do modelo SAM (vit_h) se ainda não existirem
checkpoint_path = "sam_vit_h_4b8939.pth"
if not os.path.exists(checkpoint_path):
    print("⏳ Baixando pesos do modelo SAM (vit_h)... Aguarde...")
    wget.download("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth")
    print("\n✅ Pesos baixados com sucesso!")

# 2. Inicializar o modelo na GPU (ou CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Executando no dispositivo: {device}")

sam = sam_model_registry["vit_h"](checkpoint=checkpoint_path)
sam.to(device=device)
predictor = SamPredictor(sam)

# ==============================================================================
# ETAPA 2: Carregamento Seguro da Imagem
# ==============================================================================
# Procura o ficheiro 'pluma' na pasta raiz ou dentro da pasta 'media'
arquivos = glob.glob("pluma*") + glob.glob("media/pluma*")

if not arquivos:
    raise FileNotFoundError(
        "❌ Nenhuma imagem 'pluma' encontrada! Certifique-se de que o ficheiro 'pluma.jpg' "
        "foi enviado para o Colab ou para a pasta 'media'."
    )

image_path = arquivos[0]
print(f"📁 Ficheiro de imagem carregado: '{image_path}'")

image = cv2.imread(image_path)
if image is None:
    raise ValueError(f"❌ Não foi possível ler a imagem '{image_path}'. Verifique o formato do ficheiro.")

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
h, w, _ = image_rgb.shape

# Enviar a imagem para o preditor do SAM
predictor.set_image(image_rgb)

# ==============================================================================
# ETAPA 3: Função para Exibição Lado a Lado (Corrigida)
# ==============================================================================
def show_side_by_side(image, mask, title):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # 1. Imagem Original
    axes[0].imshow(image)
    axes[0].set_title("1. Imagem Original")
    axes[0].axis('off')

    # 2. Máscara Binária
    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title("2. Máscara Binária")
    axes[1].axis('off')

    # 3. Máscara Sobreposta (Vermelha)
    overlay = image.copy()
    color_mask = np.array([255, 0, 0], dtype=np.uint8)
    overlay[mask] = (overlay[mask] * 0.5 + color_mask * 0.5).astype(np.uint8)

    axes[2].imshow(overlay)
    axes[2].set_title("3. Máscara Sobreposta")
    axes[2].axis('off')

    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ==============================================================================
# ETAPA 4: Configurações de Prompts para a Foto de Santos
# ==============================================================================

# --- CONFIGURAÇÃO 1: Apenas Pontos Positivos no Fumo / Chamas ---
input_points_c1 = np.array([
    [int(w * 0.72), int(h * 0.25)],
    [int(w * 0.75), int(h * 0.45)]
])
input_labels_c1 = np.array([1, 1])

masks, _, _ = predictor.predict(
    point_coords=input_points_c1,
    point_labels=input_labels_c1,
    multimask_output=False
)
show_side_by_side(image_rgb, masks[0], "Configuração 1: Pontos Positivos (Fumo/Chamas)")


# --- CONFIGURAÇÃO 2: Caixa Delimitadora (Bounding Box) ---
input_box_c2 = np.array([int(w * 0.58), int(h * 0.00), int(w * 0.95), int(h * 0.72)])

masks, _, _ = predictor.predict(
    box=input_box_c2,
    multimask_output=False
)
show_side_by_side(image_rgb, masks[0], "Configuração 2: Caixa Delimitadora (Bounding Box)")


# --- CONFIGURAÇÃO 3: Bounding Box + Pontos Positivos e Negativos ---
input_points_c3 = np.array([
    [int(w * 0.72), int(h * 0.25)], # Positivo: Fumo denso superior
    [int(w * 0.75), int(h * 0.45)], # Positivo: Fumo com fogo
    [int(w * 0.30), int(h * 0.20)], # Negativo: Espelho de água/baía
    [int(w * 0.20), int(h * 0.60)], # Negativo: Tanques metálicos à esquerda
    [int(w * 0.50), int(h * 0.85)]  # Negativo: Vegetação/estrada no rodapé
])
input_labels_c3 = np.array([1, 1, 0, 0, 0])

masks, _, _ = predictor.predict(
    point_coords=input_points_c3,
    point_labels=input_labels_c3,
    box=input_box_c2,
    multimask_output=False
)
show_side_by_side(image_rgb, masks[0], "Configuração 3: Caixa + Pontos Positivos e Negativos")

  Preparing metadata (setup.py) ... done
🖥️ Executando no dispositivo: cpu


FileNotFoundError: ❌ Nenhuma imagem 'pluma' encontrada! Certifique-se de que o ficheiro 'pluma.jpg' foi enviado para o Colab ou para a pasta 'media'.